[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/23.%20The%20AGV%20Dispatching%20%26%20Battery%20Management%20Problem/P23-Tier-1_executed.ipynb)

---


# Problem 23: The AGV Dispatching and Battery Management Problem

## Tier 1: Vehicle Routing Formulation

### Goal and scope

This notebook formulates a small automated guided vehicle (AGV) dispatching problem as a vehicle routing problem with time windows and battery constraints. A mixed-integer programming (MIP) model selects travel arcs, assigns each transport task to one AGV, schedules task pickups within their time windows, and allows AGVs to recharge at a charging station. The objective is to minimize total travel time.

The input is a set of tasks, AGVs, locations, travel times, energy consumptions, and charging stations. The output is a MIP status, an objective value when available, selected route arcs, and a route visualization. A small enumeration routine is retained as a fallback for simple direct routes when solving the MIP raises an exception.

### Modeling assumptions

- Each AGV starts at the common depot with its specified initial battery level. It can perform at most one depot-to-depot route in this planning horizon.
- Each task has one pickup and one drop-off. Both are visited by the same AGV, and the pickup must precede the drop-off.
- The task time window constrains arrival at its pickup. The task's service time is incurred at the pickup; no separate drop-off service time is specified.
- Travel times and energy use are deterministic and depend only on the origin and destination. The example uses Euclidean distance for both, with different conversion factors.
- Battery levels are measured after any charging at a node. Travel energy is consumed before charging at the destination. The minimum battery reserve must remain available upon arrival, including arrival back at the depot.
- Charging is possible only at listed stations. Charging amount is a decision variable, is limited by battery capacity, and takes charging amount divided by the station's charging rate in time.
- The objective includes travel time, not waiting time or charging time. Charging time affects feasible schedules through time propagation, but is not itself an objective cost.
- This static, deterministic model does not represent congestion, vehicle breakdowns, battery degradation, partial task completion, or changing information during execution.

### Solution sequence

1. Define the sets, parameters, and decision variables.
2. Build travel-time and energy-consumption matrices from the location coordinates.
3. Minimize total travel time subject to task assignment, route-flow, time-window, and battery constraints.
4. Solve the MIP with CBC and inspect the resulting routes and sensitivity runs.

A binary variable records whether an AGV uses an arc, while continuous variables represent arrival times, battery levels, and charging amounts. The MIP is the primary solution method in this tier; the fallback enumeration is deliberately restricted to simple direct routes and is not a replacement for the full MIP model.

## Mathematical model

### Sets, parameters, and variables

Let $V$ be the set of modeled locations, including depot $O$, all task pickups and drop-offs, and the charging stations. Let $K$ be the AGV set and $R$ the task set. For task $r \in R$, let $p_r$ and $d_r$ denote its pickup and drop-off, $[e_r, l_r]$ its pickup-arrival time window, and $s_r$ its pickup service time. For AGV $k \in K$, let $B_k$ be battery capacity, $B_k^{\min}$ the required reserve, and $B_k^0$ its initial battery level. Let $t_{ij}$ and $g_{ij}$ be travel time and energy use from location $i$ to location $j$. A charging station $c$ has rate $\rho_c$.

The binary variable $x_{ijk}$ is 1 when AGV $k$ travels directly from $i$ to $j$. The binary variable $y_{rk}$ is 1 when task $r$ is assigned to AGV $k$. The continuous variables $a_{ik}$ and $b_{ik}$ are, respectively, the arrival time and battery level at location $i$ for AGV $k$; $q_{ik}$ is the amount charged at location $i$ by AGV $k$. Battery level $b_{ik}$ is measured after any charge taken at $i$.

### Objective

The objective sums travel time over every selected AGV arc:

$$\min \quad \sum_{k \in K}\sum_{i \in V}\sum_{\substack{j \in V \\ j \ne i}} t_{ij}x_{ijk}.$$

Thus, the model minimizes total fleet travel time. Charging and service durations influence the schedule through constraints, but do not add directly to this objective.

### Task assignment and route flow

Each task is assigned exactly once. Its pickup and drop-off each have one incoming and one outgoing selected arc, all associated with the same AGV:

$$\sum_{k \in K}y_{rk}=1 \qquad (r \in R),$$

$$\sum_{i \in V: i \ne p_r}x_{i p_r k}=y_{rk}, \qquad
\sum_{j \in V: j \ne p_r}x_{p_r j k}=y_{rk} \qquad (r \in R, k \in K),$$

$$\sum_{i \in V: i \ne d_r}x_{i d_r k}=y_{rk}, \qquad
\sum_{j \in V: j \ne d_r}x_{d_r j k}=y_{rk} \qquad (r \in R, k \in K).$$

At every non-depot node, an AGV's number of selected incoming arcs equals its number of selected outgoing arcs. For each AGV, the number of arcs leaving the depot equals the number entering it and is at most one. Each AGV therefore has either no route or one depot-to-depot route. A charging station can be visited at most once by each AGV.

### Time windows and time propagation

The pickup arrival must lie within its time window when the task is assigned to an AGV. For a selected arc, arrival at the next non-depot location must account for travel, service at the current pickup (if any), and charging at the current station:

$$e_r-M(1-y_{rk}) \le a_{p_r k} \le l_r+M(1-y_{rk}) \qquad (r \in R, k \in K),$$

$$a_{jk} \ge a_{ik}+t_{ij}+s_i+\frac{q_{ik}}{\rho_i}-M(1-x_{ijk}) \qquad (i \ne j, j \ne O, k \in K).$$

Here $s_i=s_r$ when $i=p_r$ and is zero otherwise. The charging-time term is zero away from charging stations. A separate precedence constraint requires the drop-off time to be no earlier than pickup arrival plus pickup service and direct pickup-to-drop-off travel time. Time propagation over the selected route also enforces the order of visited locations. The depot arrival-time variable is fixed at zero and is excluded as the destination of time propagation because the model does not schedule a subsequent trip.

### Battery and charging constraints

The depot battery is fixed at its initial level. For each selected arc ending at a non-depot location, battery after charging equals battery at the previous location minus travel energy plus the charge received at the destination. The reserve must be available after travel and before charging:

$$b_{Ok}=B_k^0 \qquad (k \in K),$$

$$b_{jk}=b_{ik}-g_{ij}+q_{jk} \qquad (x_{ijk}=1, j \ne O),$$

$$b_{ik}-g_{ij} \ge B_k^{\min} \qquad (x_{ijk}=1),$$

$$B_k^{\min} \le b_{ik} \le B_k, \qquad 0 \le q_{ik}, \qquad b_{ik}+q_{ik} \le B_k.$$

The arc-conditioned equality and reserve condition are implemented with big-$M$ inequalities. Charging is zero at non-station nodes and is bounded by whether a station is visited. For a selected return-to-depot arc, the reserve condition is still enforced, even though the depot's modeled battery variable remains fixed to its initial value.

The implementation uses a sufficiently large constant $M=1000$ for this small example. In larger or differently scaled instances, big-$M$ values should be chosen from valid bounds on the relevant time and battery expressions; an unnecessarily large value can weaken the linear programming relaxation.

The MIP has a number of arc variables proportional to $|K||V|^2$, in addition to assignment and continuous variables. Its worst-case solution effort can grow rapidly as the number of locations and vehicles increases. The fallback enumeration is also intended only for very small instances.

In [ ]:
from dataclasses import dataclass
from itertools import permutations
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

## Data structures and example instance

The data classes make the model inputs explicit: a task carries its pickup, drop-off, time window, and service duration; an AGV carries its battery limits; and a charging station carries its charging rate. A location stores a two-dimensional coordinate. Euclidean distance is used to construct the example's travel-time and energy matrices.

In [ ]:
@dataclass
class Task:
    """A transport request with a pickup, drop-off, and pickup time window."""

    id: str
    pickup: str
    dropoff: str
    time_window: Tuple[float, float]
    service_time: float


@dataclass
class AGV:
    """An automated guided vehicle and its battery limits."""

    id: str
    battery_capacity: float
    battery_min: float
    initial_battery: float


@dataclass
class ChargingStation:
    """A charging location with a constant charging rate."""

    id: str
    charging_rate: float


@dataclass
class Location:
    """A terminal location represented by planar coordinates."""

    id: str
    x: float
    y: float

    def distance_to(self, other: "Location") -> float:
        """Return the Euclidean distance to another location."""
        return float(np.hypot(self.x - other.x, self.y - other.y))

In [ ]:
# The coordinates define the geometry used by both travel-time and energy calculations.
depot = Location("O", 0, 0)
p1 = Location("P1", 2, 3)
d1 = Location("D1", 5, 4)
p2 = Location("P2", 1, 5)
d2 = Location("D2", 4, 6)
c1 = Location("C1", 3, 1)

locations = {loc.id: loc for loc in [depot, p1, d1, p2, d2, c1]}

tasks = [
    Task("T1", "P1", "D1", (0, 30), 2.0),
    Task("T2", "P2", "D2", (5, 25), 1.5),
]

agvs = [
    AGV("K1", 100.0, 20.0, 100.0),
    AGV("K2", 100.0, 20.0, 100.0),
]

charging_stations = [ChargingStation("C1", 10.0)]

assert all(task.pickup in locations and task.dropoff in locations for task in tasks)
assert all(station.id in locations for station in charging_stations)
assert all(0 <= agv.battery_min <= agv.initial_battery <= agv.battery_capacity for agv in agvs)
assert all(station.charging_rate > 0 for station in charging_stations)

print("Problem setup:")
print(f"Tasks: {len(tasks)}")
print(f"AGVs: {len(agvs)}")
print(f"Charging stations: {len(charging_stations)}")
print(f"Locations: {list(locations)}")

## Travel-time and energy matrices

The model needs a travel time and an energy cost for every ordered pair of locations. In this instance, one distance unit corresponds to one minute of travel and energy use is 1.5 units per distance unit. The matrices are symmetric because Euclidean distance is symmetric; the optimization model nevertheless stores ordered pairs so that each directed travel arc has an explicit parameter.

In [ ]:
def calculate_travel_energy_matrix(
    locations: Dict[str, Location],
) -> Tuple[Dict[Tuple[str, str], float], Dict[Tuple[str, str], float]]:
    """Build directed travel-time and energy dictionaries from coordinates."""
    travel_times = {}
    energy_consumption = {}

    for loc1_id, loc1 in locations.items():
        for loc2_id, loc2 in locations.items():
            distance = loc1.distance_to(loc2)
            travel_times[(loc1_id, loc2_id)] = distance
            energy_consumption[(loc1_id, loc2_id)] = 1.5 * distance

    return travel_times, energy_consumption


travel_times, energy_consumption = calculate_travel_energy_matrix(locations)

travel_df = pd.DataFrame(
    [[travel_times[(i, j)] for j in locations] for i in locations],
    index=locations,
    columns=locations,
)
energy_df = pd.DataFrame(
    [[energy_consumption[(i, j)] for j in locations] for i in locations],
    index=locations,
    columns=locations,
)

print("Travel times (minutes):")
display(travel_df.round(2))
print("Energy consumption (energy units):")
display(energy_df.round(2))

## MIP formulation in code

The function below directly implements the mathematical model. The arc variables `x` encode route choices, and `y` links each task's pickup and drop-off to one AGV. Arrival times propagate along selected arcs, including pickup service and charging time. Battery transitions are enforced only on selected arcs by big-$M$ inequalities, while reserve constraints apply to every selected travel leg, including the return to the depot.

The returned arc list is a structured representation of the selected binary variables. It supports route extraction without relying on parsing solver-generated variable names.

In [ ]:
def solve_agv_vrp_mip(
    tasks: List[Task],
    agvs: List[AGV],
    locations: Dict[str, Location],
    travel_times: Dict[Tuple[str, str], float],
    energy_consumption: Dict[Tuple[str, str], float],
    charging_stations: List[ChargingStation],
) -> Dict:
    """Solve the AGV routing, time-window, and battery MIP with CBC."""
    depot_id = "O"
    station_by_id = {station.id: station for station in charging_stations}
    all_nodes = [depot_id]

    for task in tasks:
        all_nodes.extend([task.pickup, task.dropoff])
    all_nodes.extend(station.id for station in charging_stations)

    if len(all_nodes) != len(set(all_nodes)):
        raise ValueError("The depot, task nodes, and charging stations must be distinct.")
    if any(node not in locations for node in all_nodes):
        raise ValueError("Every modeled node must have a location.")
    if any(agv.battery_min > agv.initial_battery for agv in agvs):
        raise ValueError("An AGV's initial battery must meet its minimum reserve.")

    prob = pulp.LpProblem("AGV_VRP_Battery", pulp.LpMinimize)
    big_m = 1000.0

    # Binary arc decisions preserve the vehicle-routing structure of the model.
    x = {
        (i, j, agv.id): pulp.LpVariable(
            f"x_{i}_{j}_{agv.id}", cat="Binary"
        )
        for i in all_nodes
        for j in all_nodes
        if i != j
        for agv in agvs
    }
    y = {
        (task.id, agv.id): pulp.LpVariable(
            f"y_{task.id}_{agv.id}", cat="Binary"
        )
        for task in tasks
        for agv in agvs
    }
    arrival = {
        (node, agv.id): pulp.LpVariable(
            f"a_{node}_{agv.id}", lowBound=0, cat="Continuous"
        )
        for node in all_nodes
        for agv in agvs
    }
    battery = {
        (node, agv.id): pulp.LpVariable(
            f"b_{node}_{agv.id}",
            lowBound=agv.battery_min,
            upBound=agv.battery_capacity,
            cat="Continuous",
        )
        for node in all_nodes
        for agv in agvs
    }
    charge = {
        (node, agv.id): pulp.LpVariable(
            f"q_{node}_{agv.id}", lowBound=0, cat="Continuous"
        )
        for node in all_nodes
        for agv in agvs
    }

    prob += pulp.lpSum(
        travel_times[(i, j)] * x[(i, j, agv.id)]
        for i in all_nodes
        for j in all_nodes
        if i != j
        for agv in agvs
    ), "Total_Travel_Time"

    # Each task is assigned once, and both of its nodes use that same AGV.
    for task in tasks:
        prob += pulp.lpSum(y[(task.id, agv.id)] for agv in agvs) == 1
        for agv in agvs:
            prob += pulp.lpSum(
                x[(i, task.pickup, agv.id)]
                for i in all_nodes
                if i != task.pickup
            ) == y[(task.id, agv.id)]
            prob += pulp.lpSum(
                x[(task.pickup, j, agv.id)]
                for j in all_nodes
                if j != task.pickup
            ) == y[(task.id, agv.id)]
            prob += pulp.lpSum(
                x[(i, task.dropoff, agv.id)]
                for i in all_nodes
                if i != task.dropoff
            ) == y[(task.id, agv.id)]
            prob += pulp.lpSum(
                x[(task.dropoff, j, agv.id)]
                for j in all_nodes
                if j != task.dropoff
            ) == y[(task.id, agv.id)]

    # Flow conservation links each vehicle's visits into a route.
    for agv in agvs:
        prob += arrival[(depot_id, agv.id)] == 0
        prob += battery[(depot_id, agv.id)] == agv.initial_battery
        depot_departures = pulp.lpSum(
            x[(depot_id, j, agv.id)] for j in all_nodes if j != depot_id
        )
        depot_returns = pulp.lpSum(
            x[(i, depot_id, agv.id)] for i in all_nodes if i != depot_id
        )
        prob += depot_departures == depot_returns
        prob += depot_departures <= 1

        for node in all_nodes:
            if node != depot_id:
                prob += pulp.lpSum(
                    x[(i, node, agv.id)] for i in all_nodes if i != node
                ) == pulp.lpSum(
                    x[(node, j, agv.id)] for j in all_nodes if j != node
                )

        for station in charging_stations:
            station_visits = pulp.lpSum(
                x[(i, station.id, agv.id)]
                for i in all_nodes
                if i != station.id
            )
            prob += station_visits <= 1
            prob += charge[(station.id, agv.id)] <= (
                agv.battery_capacity * station_visits
            )

        for node in all_nodes:
            if node not in station_by_id:
                prob += charge[(node, agv.id)] == 0
            else:
                prob += (
                    battery[(node, agv.id)] + charge[(node, agv.id)]
                    <= agv.battery_capacity
                )

    # Pickup windows and task precedence connect scheduling to route choices.
    for task in tasks:
        earliest, latest = task.time_window
        for agv in agvs:
            assigned = y[(task.id, agv.id)]
            prob += arrival[(task.pickup, agv.id)] >= (
                earliest - big_m * (1 - assigned)
            )
            prob += arrival[(task.pickup, agv.id)] <= (
                latest + big_m * (1 - assigned)
            )
            prob += arrival[(task.dropoff, agv.id)] >= (
                arrival[(task.pickup, agv.id)]
                + task.service_time
                + travel_times[(task.pickup, task.dropoff)]
                - big_m * (1 - assigned)
            )

    service_time_at = {node: 0.0 for node in all_nodes}
    for task in tasks:
        service_time_at[task.pickup] = task.service_time

    # Time advances along selected arcs by travel, service, and charging duration.
    for agv in agvs:
        for i in all_nodes:
            charge_duration = (
                charge[(i, agv.id)] / station_by_id[i].charging_rate
                if i in station_by_id
                else 0
            )
            for j in all_nodes:
                if i == j or j == depot_id:
                    continue
                prob += arrival[(j, agv.id)] >= (
                    arrival[(i, agv.id)]
                    + travel_times[(i, j)]
                    + service_time_at[i]
                    + charge_duration
                    - big_m * (1 - x[(i, j, agv.id)])
                )

    # Battery changes only along selected arcs; the reserve is checked before charging.
    for agv in agvs:
        for i in all_nodes:
            for j in all_nodes:
                if i == j:
                    continue
                selected_arc = x[(i, j, agv.id)]
                energy = energy_consumption[(i, j)]

                prob += (
                    battery[(i, agv.id)] - energy
                    >= agv.battery_min - big_m * (1 - selected_arc)
                )

                if j != depot_id:
                    post_charge = charge[(j, agv.id)]
                    prob += battery[(j, agv.id)] >= (
                        battery[(i, agv.id)] - energy + post_charge
                        - big_m * (1 - selected_arc)
                    )
                    prob += battery[(j, agv.id)] <= (
                        battery[(i, agv.id)] - energy + post_charge
                        + big_m * (1 - selected_arc)
                    )

    print("Solving the MIP model with CBC...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=60))

    selected_arcs = []
    for (i, j, agv_id), variable in x.items():
        if variable.varValue is not None and variable.varValue > 0.5:
            selected_arcs.append((i, j, agv_id))

    objective_value = pulp.value(prob.objective)
    return {
        "status": pulp.LpStatus[prob.status],
        "objective": objective_value,
        "selected_arcs": selected_arcs,
        "problem": prob,
    }

## Solve the instance

The CBC solver searches for a solution to the MIP within the specified 60-second limit. A solver status is not by itself proof that a feasible route exists: route extraction and interpretation should be based on the returned status and selected arcs. If calling the MIP raises an exception, the fallback checks simple direct routes, with one task per AGV. That fallback does not model charging or combined multi-task routes and is therefore limited to small instances of this form.

In [ ]:
def enumerate_solutions(
    tasks: List[Task],
    agvs: List[AGV],
    travel_times: Dict[Tuple[str, str], float],
    energy_consumption: Dict[Tuple[str, str], float],
) -> Tuple[Optional[Tuple[str, ...]], float]:
    """Enumerate simple direct routes, assigning at most one task per AGV."""
    if len(tasks) > len(agvs):
        return None, float("inf")

    best_assignment = None
    best_cost = float("inf")

    for assignment in permutations([agv.id for agv in agvs], len(tasks)):
        total_cost = 0.0
        feasible = True

        for task, agv_id in zip(tasks, assignment):
            agv = next(vehicle for vehicle in agvs if vehicle.id == agv_id)
            route = ["O", task.pickup, task.dropoff, "O"]
            route_time_to_pickup = travel_times[("O", task.pickup)]
            route_energy = sum(
                energy_consumption[(route[index], route[index + 1])]
                for index in range(len(route) - 1)
            )

            pickup_earliest, pickup_latest = task.time_window
            if not pickup_earliest <= route_time_to_pickup <= pickup_latest:
                feasible = False
                break
            if route_energy > agv.initial_battery - agv.battery_min:
                feasible = False
                break

            total_cost += sum(
                travel_times[(route[index], route[index + 1])]
                for index in range(len(route) - 1)
            )

        if feasible and total_cost < best_cost:
            best_cost = total_cost
            best_assignment = assignment

    return best_assignment, best_cost


try:
    solution = solve_agv_vrp_mip(
        tasks,
        agvs,
        locations,
        travel_times,
        energy_consumption,
        charging_stations,
    )
except Exception as error:
    print(f"MIP solve raised an exception: {error}")
    print("Trying the restricted direct-route enumeration fallback.")
    best_assignment, best_cost = enumerate_solutions(
        tasks, agvs, travel_times, energy_consumption
    )

    fallback_arcs = []
    if best_assignment is not None:
        for task, agv_id in zip(tasks, best_assignment):
            fallback_arcs.extend(
                [
                    ("O", task.pickup, agv_id),
                    (task.pickup, task.dropoff, agv_id),
                    (task.dropoff, "O", agv_id),
                ]
            )

    solution = {
        "status": "Enumeration fallback",
        "objective": best_cost if best_assignment is not None else None,
        "selected_arcs": fallback_arcs,
        "assignment": best_assignment,
    }

print(f"Solution status: {solution['status']}")
if solution["objective"] is not None:
    print(f"Total travel time: {solution['objective']:.2f} minutes")
else:
    print("No objective value is available.")
print(f"Selected travel arcs: {len(solution['selected_arcs'])}")

## Extract routes

A route is reconstructed by following the selected outgoing arc from the depot until the AGV returns to the depot. The solver's arc decisions are directed, so route order matters. If no complete depot-to-depot route can be reconstructed, the AGV is reported as having no extracted route.

In [ ]:
def extract_routes_from_solution(
    solution: Dict,
    tasks: List[Task],
    agvs: List[AGV],
) -> Dict[str, List[str]]:
    """Reconstruct ordered node sequences from selected directed arcs."""
    del tasks  # The route is determined by the selected arcs, not task-list order.
    routes = {agv.id: [] for agv in agvs}
    arcs_by_agv = {agv.id: [] for agv in agvs}

    for from_node, to_node, agv_id in solution.get("selected_arcs", []):
        if agv_id in arcs_by_agv:
            arcs_by_agv[agv_id].append((from_node, to_node))

    for agv_id, arcs in arcs_by_agv.items():
        outgoing = {from_node: to_node for from_node, to_node in arcs}
        if "O" not in outgoing:
            continue

        route = ["O"]
        current_node = "O"
        visited = set()
        while current_node in outgoing:
            next_node = outgoing[current_node]
            if next_node in visited:
                break
            route.append(next_node)
            if next_node == "O":
                break
            visited.add(next_node)
            current_node = next_node

        if route[-1] == "O" and len(route) > 1:
            routes[agv_id] = route

    return routes


routes = extract_routes_from_solution(solution, tasks, agvs)
print("Extracted routes:")
for agv_id, route in routes.items():
    if route:
        print(f"AGV {agv_id}: {' -> '.join(route)}")
    else:
        print(f"AGV {agv_id}: no route")

## Visualize the solution

The four panels show the terminal geometry and directed routes, task-to-AGV assignment, basic fleet metrics, and route energy use. Route colors identify AGVs; node markers identify location types. The assignment and metrics panels are based on the extracted routes, so an empty route set is visible rather than being interpreted as a successful assignment.

In [ ]:
def visualize_solution(
    locations: Dict[str, Location],
    tasks: List[Task],
    agvs: List[AGV],
    routes: Dict[str, List[str]],
    solution: Dict,
    energy_consumption: Dict[Tuple[str, str], float],
):
    """Plot route geometry, task assignment, fleet metrics, and route energy."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("AGV Dispatching and Battery Management", fontsize=15, weight="bold")

    # Location markers explain the physical meaning of each plotted coordinate.
    ax1 = axes[0, 0]
    marker_styles = {
        "depot": ("ks", "Depot"),
        "pickup": ("g^", "Pickup"),
        "dropoff": ("r^", "Drop-off"),
        "charging": ("bs", "Charging station"),
    }
    legend_handles = {}
    for loc_id, loc in locations.items():
        if loc_id == "O":
            category = "depot"
        elif loc_id.startswith("P"):
            category = "pickup"
        elif loc_id.startswith("D"):
            category = "dropoff"
        elif loc_id.startswith("C"):
            category = "charging"
        else:
            continue

        marker, label = marker_styles[category]
        handle = ax1.plot(
            loc.x, loc.y, marker, markersize=9, label=label
        )[0]
        legend_handles[label] = handle
        ax1.annotate(loc_id, (loc.x, loc.y), xytext=(5, 5), textcoords="offset points")

    color_map = plt.get_cmap("tab10")
    for index, (agv_id, route) in enumerate(routes.items()):
        color = color_map(index % 10)
        for step in range(len(route) - 1):
            start = locations[route[step]]
            end = locations[route[step + 1]]
            ax1.annotate(
                "",
                xy=(end.x, end.y),
                xytext=(start.x, start.y),
                arrowprops={"arrowstyle": "->", "color": color, "alpha": 0.8},
            )
        if len(route) > 1:
            legend_handles[f"AGV {agv_id}"] = plt.Line2D(
                [0], [0], color=color, label=f"AGV {agv_id} route"
            )

    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.set_title("Terminal Layout and Routes")
    ax1.grid(True, alpha=0.3)
    if legend_handles:
        ax1.legend(handles=list(legend_handles.values()), loc="best", fontsize=8)

    # A row contains a 1 only when that task's pickup appears on the AGV route.
    ax2 = axes[0, 1]
    assignment_matrix = np.zeros((len(tasks), len(agvs)))
    for task_index, task in enumerate(tasks):
        for agv_index, agv in enumerate(agvs):
            if task.pickup in routes.get(agv.id, []):
                assignment_matrix[task_index, agv_index] = 1

    ax2.imshow(assignment_matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax2.set_xticks(range(len(agvs)))
    ax2.set_xticklabels([agv.id for agv in agvs])
    ax2.set_yticks(range(len(tasks)))
    ax2.set_yticklabels([task.id for task in tasks])
    ax2.set_xlabel("AGV")
    ax2.set_ylabel("Task")
    ax2.set_title("Task Assignment (1 = assigned)")
    for row in range(len(tasks)):
        for column in range(len(agvs)):
            ax2.text(
                column,
                row,
                f"{assignment_matrix[row, column]:.0f}",
                ha="center",
                va="center",
                color="black",
            )

    # The metrics distinguish task count from the number of utilized vehicles.
    ax3 = axes[1, 0]
    assigned_tasks = sum(
        task.pickup in route
        for task in tasks
        for route in routes.values()
    )
    utilized_agvs = sum(bool(route) for route in routes.values())
    objective = solution.get("objective")
    metric_labels = ["Tasks assigned", "AGVs utilized"]
    metric_values = [assigned_tasks, utilized_agvs]
    if objective is not None:
        metric_labels.insert(0, "Travel time (minutes)")
        metric_values.insert(0, objective)
    bars = ax3.bar(metric_labels, metric_values, color=["skyblue", "lightgreen", "salmon"][:len(metric_values)])
    ax3.set_ylabel("Value (minutes for travel time; count otherwise)")
    ax3.set_title("Solution Metrics")
    ax3.tick_params(axis="x", rotation=25)
    for bar, value in zip(bars, metric_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
        )

    # Energy use is computed along each extracted route, including depot return.
    ax4 = axes[1, 1]
    energy_by_agv = {}
    for agv_id, route in routes.items():
        if len(route) > 1:
            energy_by_agv[agv_id] = sum(
                energy_consumption[(route[index], route[index + 1])]
                for index in range(len(route) - 1)
            )

    if energy_by_agv:
        bars = ax4.bar(
            list(energy_by_agv),
            list(energy_by_agv.values()),
            color="orange",
            label="Route energy use",
        )
        ax4.set_xlabel("AGV")
        ax4.set_ylabel("Energy units")
        ax4.set_title("Energy Use by AGV")
        ax4.legend()
        for bar, value in zip(bars, energy_by_agv.values()):
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:.1f}",
                ha="center",
                va="bottom",
            )
    else:
        ax4.text(
            0.5,
            0.5,
            "No extracted AGV routes",
            ha="center",
            va="center",
            transform=ax4.transAxes,
        )
        ax4.set_title("Energy Use by AGV")

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_solution(
    locations,
    tasks,
    agvs,
    routes,
    solution,
    energy_consumption,
)

## Sensitivity analysis

Sensitivity analysis reruns the same MIP while changing only the AGVs' battery capacity and initial charge. The tested capacity values are the experiment settings; they are not changes to the main instance. Comparing solver status and objective across these runs shows whether battery capacity affects feasibility or the travel-time optimum for this particular data set. The charging-station and time-window remarks below are qualitative model interpretations, not additional computed experiments.

In [ ]:
def sensitivity_analysis(
    tasks: List[Task],
    agvs: List[AGV],
    locations: Dict[str, Location],
    travel_times: Dict[Tuple[str, str], float],
    energy_consumption: Dict[Tuple[str, str], float],
    charging_stations: List[ChargingStation],
) -> List[Dict]:
    """Rerun the MIP for the specified battery-capacity values."""
    battery_capacities = [60, 80, 100, 120, 140]
    results_battery = []

    for capacity in battery_capacities:
        test_agvs = [
            AGV(agv.id, capacity, agv.battery_min, capacity)
            for agv in agvs
        ]
        try:
            test_solution = solve_agv_vrp_mip(
                tasks,
                test_agvs,
                locations,
                travel_times,
                energy_consumption,
                charging_stations,
            )
            results_battery.append(
                {
                    "capacity": capacity,
                    "status": test_solution["status"],
                    "objective": test_solution["objective"],
                }
            )
        except Exception as error:
            results_battery.append(
                {
                    "capacity": capacity,
                    "status": f"Failed: {error}",
                    "objective": None,
                }
            )

    print("Battery-capacity sensitivity:")
    for result in results_battery:
        objective = result["objective"]
        objective_text = "unavailable" if objective is None else f"{objective:.2f}"
        print(
            f"Capacity: {result['capacity']:>3} | "
            f"Status: {result['status']} | "
            f"Travel time: {objective_text}"
        )

    print("\nCharging-station interpretation:")
    print(
        "The model permits charging at listed stations; whether a station is "
        "used depends on route feasibility and the travel-time objective."
    )
    print("\nTime-window interpretation:")
    print(
        "Pickup windows constrain feasible schedules; tighter windows can "
        "remove otherwise feasible routes."
    )
    return results_battery


sensitivity_results = sensitivity_analysis(
    tasks,
    agvs,
    locations,
    travel_times,
    energy_consumption,
    charging_stations,
)

## Summary and interpretation

The MIP expresses dispatching as a coordinated choice of task assignments and directed travel arcs. Assignment constraints ensure that each task's pickup and drop-off belong to the same AGV. Flow constraints connect visited nodes into depot-based routes. Time constraints enforce pickup windows and account for travel, pickup service, and charging duration. Battery constraints enforce energy consumption, charging limits, capacity, and the minimum reserve.

Interpret the solver status before treating any objective value or route as optimal. A status of `Optimal` indicates that the solver reports an optimal solution for the encoded MIP; other statuses should be interpreted according to the solver's termination condition. The visualization describes the extracted routes, while the sensitivity table records the results of separate capacity runs. A charging station may be available without appearing in a route: availability does not imply that charging is necessary or beneficial for the travel-time objective.

This tier provides an exact mathematical formulation for a small deterministic instance and a basis for understanding the problem's structure. The model does not include dynamic dispatching or operational uncertainties, and MIP solution effort can increase substantially as the instance grows. The restricted enumeration fallback checks only simple direct routes and does not represent the full MIP's charging or multi-task routing decisions.